# I Have The Power! Interactive Bokeh Scatter Plot
**Data set:** Chick-fil-A nutrition (`chick-fil-a.csv`), 290 menu items

The scatter plot compares **Calories** (x-axis) to **Protein** (y-axis), with point color showing **Sodium**. Two widgets filter the data:
1. **Slider:** maximum calories
2. **Range slider:** sodium range (minimum and maximum)

No tooltip is included, so classmates can add one in the response post. All columns are kept in the `ColumnDataSource` with simple names (for example `@sugar` or `@fat`) to make that easy.

In [1]:
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, output_file, save
from bokeh.models import (ColumnDataSource, Slider, RangeSlider, CustomJS, Div,
                          LinearColorMapper, ColorBar)
from bokeh.palettes import Viridis256
from bokeh.layouts import column, row

output_notebook()

Loading BokehJS ...

## Load and clean the data

In [2]:
df = pd.read_csv('chick-fil-a.csv')

# Clean the item names (remove non-breaking spaces and extra whitespace)
df['Menu'] = df['Menu'].str.replace('\u00a0', ' ').str.replace(r'\s+', ' ', regex=True).str.strip()

# Rename columns to simple names that work well in Bokeh (and in tooltips like @sugar)
df = df.rename(columns={
    'Menu': 'item', 'Calories': 'calories', 'Fat (G)': 'fat', 'Sat. Fat (G)': 'sat_fat',
    'Trans Fat (G)': 'trans_fat', 'Cholesterol (MG)': 'cholesterol', 'Sodium (MG)': 'sodium',
    'Carbohydrates (G)': 'carbs', 'Fiber (G)': 'fiber', 'Sugar (G)': 'sugar', 'Protein (G)': 'protein'})

print(df.shape)
print('Missing values:', df.isna().sum().sum(), '| Duplicate rows:', df.duplicated().sum())
df.describe().round(1)

(290, 11)
Missing values: 0 | Duplicate rows: 0


,calories,fat,sat_fat,trans_fat,cholesterol,sodium,carbs,fiber,sugar,protein
count,290.0,290.0,290.0,290.0,290.0,290.0,290.0,290.0,290.0,290.0
mean,435.8,20.8,7.0,0.1,74.4,842.3,46.6,2.9,27.2,17.4
std,766.0,47.8,25.8,0.6,141.7,1944.0,83.9,7.0,68.1,34.1
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,110.0,1.1,0.0,0.0,0.0,76.2,9.2,0.0,1.0,1.0
50%,270.0,9.0,2.5,0.0,25.0,395.0,28.0,1.0,7.0,10.0
75%,540.0,26.0,8.0,0.0,85.0,1145.0,44.8,3.0,26.0,28.0
max,9850.0,624.0,355.0,7.0,1500.0,26050.0,616.0,63.0,517.0,442.0


## Build the interactive scatter plot

In [3]:
CAL_DEFAULT = 1500   # hides 10 catering trays and gallon drinks on first load

full_source = ColumnDataSource(df)                               # never changes
view_source = ColumnDataSource(df[df['calories'] <= CAL_DEFAULT]) # what the plot shows

color_mapper = LinearColorMapper(palette=Viridis256[::-1], low=0, high=2500)

p = figure(title='Chick-fil-A Menu: Calories vs. Protein (color = sodium)',
           width=850, height=550,
           x_axis_label='Calories', y_axis_label='Protein (g)',
           tools='pan,wheel_zoom,box_zoom,reset,save')
p.scatter('calories', 'protein', source=view_source, size=10, alpha=0.75,
          line_color='white', line_width=0.6,
          fill_color={'field': 'sodium', 'transform': color_mapper})

color_bar = ColorBar(color_mapper=color_mapper, title='Sodium (mg); 2,500+ is darkest', width=12)
p.add_layout(color_bar, 'right')
p.title.text_font_size = '15px'
p.xgrid.grid_line_alpha = 0.3
p.ygrid.grid_line_alpha = 0.3

# Widget 1: maximum calories (slider)
cal_slider = Slider(title='Maximum calories', start=0, end=int(df['calories'].max()),
                    value=CAL_DEFAULT, step=50, width=400)

# Widget 2: sodium range (range slider)
sodium_slider = RangeSlider(title='Sodium range (mg)', start=0, end=int(df['sodium'].max()),
                            value=(0, int(df['sodium'].max())), step=50, width=400)

count_div = Div(text=f'<b>Showing {len(view_source.data["item"])} of {len(df)} menu items</b>')

callback = CustomJS(args=dict(full=full_source, view=view_source, cal=cal_slider,
                              sod=sodium_slider, div=count_div, total=len(df)), code='''
    const d = full.data;
    const maxCal = cal.value;
    const [lowSod, highSod] = sod.value;
    const out = {};
    for (const key of Object.keys(d)) { out[key] = []; }
    for (let i = 0; i < d['calories'].length; i++) {
        if (d['calories'][i] <= maxCal && d['sodium'][i] >= lowSod && d['sodium'][i] <= highSod) {
            for (const key of Object.keys(d)) { out[key].push(d[key][i]); }
        }
    }
    view.data = out;
    div.text = '<b>Showing ' + out['calories'].length + ' of ' + total + ' menu items</b>';
''')
cal_slider.js_on_change('value', callback)
sodium_slider.js_on_change('value', callback)

layout = column(row(cal_slider, sodium_slider), count_div, p)
show(layout)

## Save a standalone HTML copy
The HTML file opens in any browser and keeps the widgets working.

In [4]:
output_file('chickfila_bokeh_scatter.html', title='Chick-fil-A Calories vs. Protein')
save(layout)

'C:\\Users\\jagis\\projects\\I Have The Power - Bokeh Notebook\\chickfila_bokeh_scatter.html'

## Supporting numbers used in the write-up

In [5]:
items = df[df['calories'] <= CAL_DEFAULT].copy()
print('Items shown at default (<= 1,500 calories):', len(items))
print('Correlation, calories vs. protein (<= 1,500 cal):', round(items['calories'].corr(items['protein']), 2))
print('Correlation, calories vs. protein (all items):  ', round(df['calories'].corr(df['protein']), 2))

items['protein_per_100_cal'] = items['protein'] / items['calories'] * 100
print('\nMost protein per 100 calories (items with 100+ calories):')
print(items[items['calories'] >= 100].nlargest(5, 'protein_per_100_cal')[['item', 'calories', 'protein', 'sodium', 'protein_per_100_cal']].round(1).to_string(index=False))

print('\nItems with 400+ calories and the least protein:')
print(items[items['calories'] >= 400].nsmallest(6, 'protein')[['item', 'calories', 'protein', 'sugar']].to_string(index=False))

print('\nHighest-sodium items (<= 1,500 cal):')
print(items.nlargest(6, 'sodium')[['item', 'calories', 'protein', 'sodium']].to_string(index=False))

print('\nItems above 1,500 calories:')
print(df[df['calories'] > CAL_DEFAULT][['item', 'calories', 'protein', 'sodium']].to_string(index=False))

Items shown at default (<= 1,500 calories): 280
Correlation, calories vs. protein (<= 1,500 cal): 0.63
Correlation, calories vs. protein (all items):   0.88

Most protein per 100 calories (items with 100+ calories):
                 item  calories  protein  sodium  protein_per_100_cal
      Grilled Nuggets       130       25     440                 19.2
 8 ct Grilled Nuggets       130       25     440                 19.2
        Grilled Filet       110       21     370                 19.1
12 ct Grilled Nuggets       200       38     660                 19.0
 Chick-fil-A® Nuggets       250       27    1210                 10.8

Items with 400+ calories and the least protein:
                                                          item  calories  protein  sugar
 Gallon Chick-fil-A® Iced Tea (1/2 Sweet Tea, 1/2 Unsweet Tea)       770        0    200
                Gallon Sunjoy® (1/2 Unsweet Tea, 1/2 Lemonade)      1040        0    271
Gallon Chick-fil-A® Lemonade (1/2 Lemonade, 1/2 